In [6]:
import shutil
import os

# Clear all checkpoints
shutil.rmtree("/tmp/checkpoints", ignore_errors=True)
print("Checkpoints cleared")

# Drop iceberg tables so they get recreated fresh
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.bronze")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.silver")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.gold")
print("Tables dropped")

Checkpoints cleared
Tables dropped


In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")


Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP
) USING iceberg
""")

DataFrame[]

In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [4]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [5]:
# SEMINAR TASK 
# Consume a few messages from the topic using kafka-console-consumer.sh to verify they are there

# docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic taxi-trips --from-beginning  --max-messages 5"


In [7]:
# starts the query
bronze = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp")
    )
)

query = (
    bronze.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/taxi_bronze")
    .toTable("lakehouse.taxi.bronze")
)

query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/conda/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart

KeyboardInterrupt: 

In [ ]:
# RUN THIS TO STOP THE QUERY
query.stop()

In [8]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 10").show()

+--------+
|count(1)|
+--------+
|     540|
+--------+

+---------+--------------------+----------+---------+------+--------------------+
|kafka_key|           raw_value|     topic|partition|offset|     kafka_timestamp|
+---------+--------------------+----------+---------+------+--------------------+
|        1|{"VendorID": 1, "...|taxi-trips|        0|   139|2026-04-04 17:18:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|   140|2026-04-04 17:18:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|   141|2026-04-04 17:18:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|   142|2026-04-04 17:18:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|   143|2026-04-04 17:18:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|   144|2026-04-04 17:18:...|
|        2|{"VendorID": 2, "...|taxi-trips|        2|   356|2026-04-04 17:18:...|
|        2|{"VendorID": 2, "...|taxi-trips|        2|   357|2026-04-04 17:18:...|
|        2|{"VendorID": 2, "...|taxi-trips

In [16]:
# === SILVER LAYER ===
spark.sql("""
    CREATE OR REPLACE TABLE lakehouse.taxi.silver (
        VendorID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count DOUBLE,
        trip_distance DOUBLE,
        RatecodeID INT,
        store_and_fwd_flag BOOLEAN,
        PULocationID INT,
        DOLocationID INT,
        payment_type INT,
        fare_amount DOUBLE,
        extra DOUBLE,
        mta_tax DOUBLE,
        tip_amount DOUBLE,
        tolls_amount DOUBLE,
        improvement_surcharge DOUBLE,
        total_amount DOUBLE,
        congestion_surcharge DOUBLE,
        Airport_fee DOUBLE,
        cbd_congestion_fee DOUBLE,
        trip_duration_minutes INT,
        avg_speed_kmh DOUBLE,
        pickup_zone STRING,
        pickup_borough STRING,
        dropoff_zone STRING,
        dropoff_borough STRING,
        is_peak_hour BOOLEAN,
        kafka_timestamp TIMESTAMP
    ) USING iceberg
""")

trip_schema = """
    VendorID INT, tpep_pickup_datetime TIMESTAMP, tpep_dropoff_datetime TIMESTAMP,
    passenger_count DOUBLE, trip_distance DOUBLE, RatecodeID DOUBLE,
    store_and_fwd_flag STRING, PULocationID INT, DOLocationID INT,
    payment_type LONG, fare_amount DOUBLE, extra DOUBLE, mta_tax DOUBLE,
    tip_amount DOUBLE, tolls_amount DOUBLE, improvement_surcharge DOUBLE,
    total_amount DOUBLE, congestion_surcharge DOUBLE, Airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE
"""

# Load zones
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones_cleaned = (zones
    .filter(F.col("LocationID").isNotNull() & (F.col("LocationID") > 0))
    .filter(F.col("Borough").isNotNull() & (F.col("Borough") != ""))
    .filter(F.col("Zone").isNotNull() & (F.col("Zone") != ""))
    .dropDuplicates(["LocationID"])
)

# Parse from bronze
silver_df = (
    spark.table("lakehouse.taxi.bronze")
    .select(F.from_json("raw_value", trip_schema).alias("d"), "kafka_timestamp")
    .select("d.*", "kafka_timestamp")
)

# Cast types
silver_df = (silver_df
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag") == "Y")
)

# Clean
silver_df = (silver_df
    .filter((F.col("passenger_count").isNotNull()) & (F.col("passenger_count") > 0))
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter(F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99]))
    .filter(F.col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]))
    .withColumn("trip_duration_minutes",
        ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60).cast("int"))
    .filter((F.col("trip_duration_minutes") > 0) & (F.col("trip_duration_minutes") < 1440))
    .withColumn("avg_speed_kmh",
        F.try_divide(F.col("trip_distance") * 1.60934, F.col("trip_duration_minutes") / 60))
    .filter((F.col("avg_speed_kmh") >= 2) & (F.col("avg_speed_kmh") <= 130))
    .dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])
)

# Enrich with zones
silver_df = (silver_df
    .join(zones_cleaned.select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough")
    ), on="PULocationID", how="left")
    .join(zones_cleaned.select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("Borough").alias("dropoff_borough")
    ), on="DOLocationID", how="left")
)

# Write to silver
silver_df.writeTo("lakehouse.taxi.silver").append()

bronze_cnt = spark.table("lakehouse.taxi.bronze").count()
silver_cnt = spark.table("lakehouse.taxi.silver").count()
print(f"Bronze: {bronze_cnt} rows")
print(f"Silver: {silver_cnt} rows ({bronze_cnt - silver_cnt} bad records filtered)")
spark.sql("SELECT * FROM lakehouse.taxi.silver LIMIT 5").show(truncate=False)

Bronze: 901 rows
Silver: 865 rows (36 bad records filtered)
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+------------------+-----------------------------+--------------+---------------------+---------------+------------+-----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|avg_speed_kmh     |pickup_zone                  |pickup_borough|dropoff_zone         |dropoff_borough|is_peak_hour|kafka_timestamp        |
+--------+--------------------+-------

In [22]:
# GOLD pole obvs neid veel testinud, lihtsalt panen esialgse lahenduse üles, et pärast ei peaks mergega dealima

# VARIANT A: streaming 

# spark.sql("""
# CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
#     day DATE,
#     pickup_zone STRING,
#     trip_count LONG,
#     avg_distance DOUBLE,
#     avg_fare DOUBLE,
#     avg_total DOUBLE,
#     tip_rate_pct DOUBLE,
#     total_revenue DOUBLE
#     ) USING iceberg
# PARTITIONED BY (day)
# """)

# gold_source = (
#     spark.readStream
#     .format("iceberg")
#     .option("stream-from-timestamp", "0")
#     .load("lakehouse.taxi.silver")
# )

# gold = (
#     gold_source
#     .withColumn("day", F.to_date("tpep_pickup_datetime"))
#     .groupBy("day", "pickup_zone")
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
#         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
#         F.round(F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0))
#                 / F.count("*") * 100, 2).alias("tip_rate_pct"),
#         F.round(F.avg("total_amount"), 2).alias("avg_total"),
#         F.round(F.sum("total_amount"), 2).alias("total_revenue"),
#     )
# )

#def upsert_gold(batch_df, batch_id):
#    batch_df.createOrReplaceTempView("gold_batch")
#    spark.sql("""
#        MERGE INTO lakehouse.taxi.gold AS target
#        USING gold_batch AS source
#        ON target.day = source.day AND target.pickup_zone = source.pickup_zone
#        WHEN MATCHED THEN UPDATE SET *
#        WHEN NOT MATCHED THEN INSERT *
#    """)
    
#query_gold = (
#    gold_stream.writeStream
#    .outputMode("update")     
#    .trigger(processingTime="1 minute")
#    .option("checkpointLocation", "s3://checkpoints/gold") see vaja ülev vaadata!! 
#    .foreachBatch(upsert_gold)
#    .start()
#)

# query_gold.awaitTermination()


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1488384012.py, line 52)

In [ ]:
# GOLD

# VARIANT B: BATCH 

# spark.sql("""
# CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
#     day DATE,
#     pickup_zone STRING,
#     trip_count LONG,
#     avg_distance DOUBLE,
#     avg_fare DOUBLE,
#     avg_total DOUBLE,
#     tip_rate_pct DOUBLE,
#     total_revenue DOUBLE
#     ) USING iceberg
# PARTITIONED BY (day)
# """)

# gold_df = (
#     spark.table("lakehouse.taxi.silver")
#     .withColumn("day", F.to_date("tpep_pickup_datetime"))
#     # hourly: .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime"))
#     .groupBy("day", "pickup_zone")
#     # hourly: .groupBy("hour", "pickup_zone")
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
#         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
#         F.round(F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0)).cast(DoubleType())
#                 / F.count("*") * 100, 2).alias("tip_rate_pct"),
#         F.round(F.avg("total_amount"), 2).alias("avg_total"),
#         F.round(F.sum("total_amount"), 2).alias("total_revenue"),
#     )
# )


# todo mõelda, kas mõistlikum teha hourly või daily, kui hour, siis hour TIMESTAMP. kui teha tunni kaupa, siis .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime")) ja .groupBy("hour", "pickup_zone")
# gold_df.writeTo("lakehouse.taxi.gold").append()
